# PV fault detection — Random Forest (RF)

**Model settings:** Paper settings (Table 3): 1000 trees, entropy criterion, no depth limit, min_samples_split 2, min_samples_leaf 1.

**Data (all simulated):**
- *System 1 initial:* the paper's 4 features + Isc/G
- *System 1 final:* + I-V curve features (Vmp, Imp, FF) from the high-resolution sensor, + Isc/G and the three physics ratios
- *System 2:* trained on System 1, tested on System 2 (zero-shot and few-shot), 4 features + Isc/G

Every result is on rows the model never saw: 80 % train / 20 % test, 5 runs (like the paper).

In [1]:
# Where the data files are
import sys
if 'google.colab' in sys.modules:                 # running in Google Colab
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/PV_Project/PV_final'
else:                                             # running on your computer (VS Code / Jupyter)
    DATA_DIR = 'data'


## Load the data and build the inputs

In [2]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier

# Simulated datasets (made by pv_simulator_v2.py). Fault: 0 = normal, 1 = partial shading, 2 = dirt
s1_initial = pd.read_csv(f'{DATA_DIR}/S1_sim500_realistic_4features.csv')   # System 1: paper's 4 features
s1_final   = pd.read_csv(f'{DATA_DIR}/S1_sim500_realistic_hires_IV.csv')    # System 1: + I-V curve, high-res sensor
s2         = pd.read_csv(f'{DATA_DIR}/S2_sim200_realistic_4features.csv')   # System 2: paper's 4 features

def features(df):
    """Model inputs = the measured columns + the physics features they allow."""
    X = df.drop(columns='Fault').copy()
    X['Isc/G'] = X['Isc/MaxIsc'] / X['G/1000']                 # current per unit of sunlight
    if 'Vmp/MaxVoc' in X:                                       # I-V dataset: three more physics ratios
        X['Vmp/Voc'] = X['Vmp/MaxVoc'] / X['Voc/MaxVoc']        # shading moves the curve's knee
        X['Imp/Isc'] = X['Imp/MaxIsc'] / X['Isc/MaxIsc']        # shading: MPP current falls below Isc
        X['Imp/G']   = X['Imp/MaxIsc'] / X['G/1000']            # dirt: less MPP current per unit sunlight
    return X

LABELS = ['normal', 'shading', 'dirt']


## The model

In [3]:
def train_model(X, y, seed, weights=None):
    model = RandomForestClassifier(n_estimators=1000, criterion='entropy', max_depth=None,
                                   min_samples_split=2, min_samples_leaf=1, random_state=seed, n_jobs=-1)
    return model.fit(X, y, sample_weight=weights)

## How each experiment is scored (5 runs)

In [4]:
def evaluate(data, title):
    """5 runs, each with a new 80/20 split (like the paper). Prints every run and the average."""
    X, y = features(data), data.Fault
    rows, matrices, reports = [], [], []
    for run in range(5):
        tr, te = train_test_split(X.index, test_size=0.2, stratify=y, random_state=run)
        model = train_model(X.loc[tr], y[tr], seed=run)
        pred = model.predict(X.loc[te])
        rows.append(dict(Run=run + 1, Accuracy=accuracy_score(y[te], pred),
                         Precision=precision_score(y[te], pred, average='weighted'),
                         Recall=recall_score(y[te], pred, average='weighted'),
                         F1=f1_score(y[te], pred, average='weighted')))
        matrices.append(confusion_matrix(y[te], pred))
        reports.append(classification_report(y[te], pred, target_names=LABELS, digits=4))
        print(f'Run {run + 1}: accuracy = {rows[-1]["Accuracy"]:.4f}')
    res = pd.DataFrame(rows).set_index('Run')
    print(f'\n===== {title} =====')
    print(res.round(4).to_string())
    print(f'\nMean of 5 runs:  ' + '   '.join(f'{m} = {res[m].mean():.4f}' for m in res.columns)
          + f'   (accuracy std = {res.Accuracy.std():.4f})')
    best = (res.Accuracy - res.Accuracy.mean()).abs().idxmin()          # the paper shows the run closest to the mean
    print(f'\nConfusion matrix, run {best} (closest to the mean). Rows = true class, columns = predicted:')
    print(pd.DataFrame(matrices[best - 1], index=['true ' + l for l in LABELS], columns=['pred ' + l for l in LABELS]).to_string())
    print(f'\nClassification report, run {best}:\n{reports[best - 1]}')
    return res.mean()


## System 1 — initial (4 features + Isc/G)

In [5]:
s1_init = evaluate(s1_initial, 'System 1 initial: 4 features + Isc/G')

Run 1: accuracy = 0.8000


Run 2: accuracy = 0.8000


Run 3: accuracy = 0.8400


Run 4: accuracy = 0.8100


Run 5: accuracy = 0.7800

===== System 1 initial: 4 features + Isc/G =====
     Accuracy  Precision  Recall      F1
Run                                     
1        0.80     0.8000    0.80  0.8000
2        0.80     0.7984    0.80  0.7983
3        0.84     0.8446    0.84  0.8372
4        0.81     0.8071    0.81  0.8077
5        0.78     0.7793    0.78  0.7738

Mean of 5 runs:  Accuracy = 0.8060   Precision = 0.8059   Recall = 0.8060   F1 = 0.8034   (accuracy std = 0.0219)

Confusion matrix, run 4 (closest to the mean). Rows = true class, columns = predicted:
              pred normal  pred shading  pred dirt
true normal            34             0          0
true shading            0            23         10
true dirt               2             7         24

Classification report, run 4:
              precision    recall  f1-score   support

      normal     0.9444    1.0000    0.9714        34
     shading     0.7667    0.6970    0.7302        33
        dirt     0.7059    0.7273    

## System 1 — final (I-V + high-res sensor + physics features)

In [6]:
s1_fin = evaluate(s1_final, 'System 1 final: I-V + hi-res sensor + Isc/G + physics ratios')

Run 1: accuracy = 0.9700


Run 2: accuracy = 0.9800


Run 3: accuracy = 0.9800


Run 4: accuracy = 0.9900


Run 5: accuracy = 0.9500

===== System 1 final: I-V + hi-res sensor + Isc/G + physics ratios =====
     Accuracy  Precision  Recall      F1
Run                                     
1        0.97     0.9703    0.97  0.9700
2        0.98     0.9811    0.98  0.9800
3        0.98     0.9811    0.98  0.9800
4        0.99     0.9903    0.99  0.9900
5        0.95     0.9564    0.95  0.9497

Mean of 5 runs:  Accuracy = 0.9740   Precision = 0.9758   Recall = 0.9740   F1 = 0.9739   (accuracy std = 0.0152)

Confusion matrix, run 1 (closest to the mean). Rows = true class, columns = predicted:
              pred normal  pred shading  pred dirt
true normal            33             0          1
true shading            0            33          0
true dirt               2             0         31

Classification report, run 1:
              precision    recall  f1-score   support

      normal     0.9429    0.9706    0.9565        34
     shading     1.0000    1.0000    1.0000        33
        dirt 

## System 2 — zero-shot and few-shot
Zero-shot: trained on System 1 only. Few-shot k: k System 2 rows per class added to training.

In [7]:
X1, y1 = features(s1_initial), s1_initial.Fault
X2, y2 = features(s2), s2.Fault

# Zero-shot: trained on System 1 only, tested on all 200 System 2 rows (5 runs)
zero = []
for run in range(5):
    model = train_model(X1, y1, seed=run)
    pred = model.predict(X2)
    zero.append(accuracy_score(y2, pred))
    print(f'Zero-shot run {run + 1}: accuracy = {zero[-1]:.4f}')
print(f'Zero-shot mean = {np.mean(zero):.4f}')
print('Confusion matrix of the last run (rows = true, columns = predicted):')
print(pd.DataFrame(confusion_matrix(y2, pred), index=['true ' + l for l in LABELS], columns=['pred ' + l for l in LABELS]).to_string())

# Few-shot: add k System 2 rows per class to training (weighted 5x), test on the other System 2 rows (5 runs)
few = {}
for k in [5, 10]:
    accs = []
    for run in range(5):
        shots = s2.groupby('Fault', group_keys=False).sample(k, random_state=100 * k + run)
        rest = s2.drop(shots.index)
        X = pd.concat([X1, features(shots)], ignore_index=True)
        y = pd.concat([y1, shots.Fault], ignore_index=True)
        w = np.r_[np.ones(len(X1)), np.full(len(shots), 5.0)]
        model = train_model(X, y, seed=run, weights=w)
        accs.append(accuracy_score(rest.Fault, model.predict(features(rest))))
    few[k] = np.mean(accs)
    print(f'\nFew-shot k={k}: ' + ', '.join(f'{a:.4f}' for a in accs) + f'   mean = {few[k]:.4f}')


Zero-shot run 1: accuracy = 0.8050


Zero-shot run 2: accuracy = 0.8050


Zero-shot run 3: accuracy = 0.8050


Zero-shot run 4: accuracy = 0.8050


Zero-shot run 5: accuracy = 0.8050
Zero-shot mean = 0.8050
Confusion matrix of the last run (rows = true, columns = predicted):
              pred normal  pred shading  pred dirt
true normal            67             0          0
true shading           38            28          1
true dirt               0             0         66



Few-shot k=5: 0.9946, 0.9946, 0.9946, 0.9676, 0.9946   mean = 0.9892



Few-shot k=10: 0.9941, 1.0000, 0.9941, 0.9941, 0.9941   mean = 0.9953


## Summary

In [8]:
summary = pd.DataFrame({
    'System 1 initial (4 features + Isc/G)': [s1_init['Accuracy']],
    'System 1 final (I-V + hi-res + physics)': [s1_fin['Accuracy']],
    'System 2 zero-shot': [np.mean(zero)],
    'System 2 few-shot k=5': [few[5]],
    'System 2 few-shot k=10': [few[10]],
}, index=['RF']).round(4)
print(summary.T.to_string())
print('\nPaper (real data): System 1 accuracy 0.9567 | System 2 zero-shot 0.5400')


                                             RF
System 1 initial (4 features + Isc/G)    0.8060
System 1 final (I-V + hi-res + physics)  0.9740
System 2 zero-shot                       0.8050
System 2 few-shot k=5                    0.9892
System 2 few-shot k=10                   0.9953

Paper (real data): System 1 accuracy 0.9567 | System 2 zero-shot 0.5400
